# 18C — Build Cycle-25 SHARP Test Arrays

## Purpose

Construct the **independent 2021–2025 Cycle-25 SHARP test arrays** using the already-defined temporal-history design and the same 15 magnetic features used in 18A/18B.

This notebook performs **data assembly only**. It does not fit, calibrate, tune, or evaluate any model.

## Locked scientific contract

- Forecast target: same-active-region M/X flare occurrence in `(t, t+48 h]`
- Independent test interval: 2021–2025
- History slots: t−288, t−192, t−96 minutes
- SHARP features: 15 ordered magnetic parameters
- No Cycle-25 sample may influence training, calibration, threshold selection, feature selection, or model choice.

## Quality rule

The same conservative rule used in 18A/18B is applied:

- QUALITY = 0 at all three SHARP source records
- 15/15 finite features at each record
- exactly one source match
- no explicit NOAA identity conflict
- three unique exact historical records

## Important limitation

This notebook preserves the existing label and timing status. It does not repair labels or certify historical availability.


In [ ]:
from pathlib import Path
import ast, csv, gzip, json, math, re
from collections import defaultdict
import numpy as np
import pandas as pd

HOME = Path.home()
META = HOME / "aia17_metadata_stage1"
OUT = HOME / "aia_sharp_cycle25_input"
OUT.mkdir(parents=True, exist_ok=True)

FEATURES = [
    "MEANGBZ","MEANGAM","MEANGBT","MEANGBH","MEANJZD",
    "TOTUSJZ","MEANALP","MEANJZH","ABSNJZH","SAVNCPP",
    "MEANSHR","SHRGT45","R_VALUE","USFLUX","TOTPOT",
]
LAGS = [288, 192, 96]

print("META:", META)
print("OUT:", OUT)


## 1. Resolve exact existing inputs

In [ ]:
# Existing temporal index and broad-role proposal are reused.
temporal_reports = sorted((META / "temporal_manifest_v1" / "reports").glob("*"))
broad_reports = sorted((META / "broad_cycle24_finalfit_v2" / "reports").glob("*"))

if not temporal_reports:
    raise RuntimeError("No temporal_manifest_v1 report directory found.")
if not broad_reports:
    raise RuntimeError("No broad_cycle24_finalfit_v2 report directory found.")

TEMP = temporal_reports[-1]
BROAD = broad_reports[-1]

cand_path = TEMP / "temporal_sequence_candidates.jsonl.gz"
role_path = BROAD / "broad_cycle24_final_role_candidates.csv.gz"

if not cand_path.exists():
    raise RuntimeError(f"Missing {cand_path}")
if not role_path.exists():
    raise RuntimeError(f"Missing {role_path}")

# Resolve the exact cached raw SHARP source already used by the Cycle-24 array builder.
cache_candidates = sorted((META / "cache").glob("sharp96_*.csv"))
if not cache_candidates:
    raise RuntimeError("No cached sharp96 CSV found.")
sharp_path = cache_candidates[-1]

print("Temporal candidates:", cand_path)
print("Role table:", role_path)
print("Raw SHARP source:", sharp_path)


## 2. Select only the independent 2021–2025 test targets

In [ ]:
roles = pd.read_csv(role_path)
print("Role columns:", list(roles.columns))

role_col = None
for c in ["proposed_final_role", "role", "proposed_role"]:
    if c in roles.columns:
        role_col = c
        break
if role_col is None:
    raise RuntimeError("Could not identify role column.")

test = roles[roles[role_col].eq("independent_cycle25_test")].copy()
if test.empty:
    raise RuntimeError("No independent_cycle25_test rows found.")

year_col = "stored_year" if "stored_year" in test.columns else None
if year_col is not None:
    test = test[test[year_col].between(2021, 2025)]

sid_col = "target_sample_id"
label_col = "original_label_48h_final"
required = [sid_col, label_col, "HARPNUM", "NOAA_AR_clean", "region_component_id"]
missing = [c for c in required if c not in test.columns]
if missing:
    raise RuntimeError(f"Missing required role columns: {missing}")

print("Independent test targets:", len(test))
print("Positives:", int(test[label_col].sum()))
print("Regions:", test["region_component_id"].nunique())
print("Years:", sorted(test[year_col].unique()) if year_col else "not available")


## 3. Read the existing three-slot temporal history for those test targets

In [ ]:
selected = set(test[sid_col])
frames_by_target = {}
target_meta = {}

with gzip.open(cand_path, "rt", encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        sid = rec["target_sample_id"]
        if sid not in selected:
            continue
        if rec.get("history_status") != "NOMINAL_HISTORY_OBJECTS_AVAILABLE_TIMING_PENDING":
            continue
        frames = rec.get("frames", [])
        if len(frames) != 3 or [x.get("lag_minutes") for x in frames] != LAGS:
            continue
        if not all(x.get("nominal_candidate") is True for x in frames):
            continue

        keys = []
        for frame in frames:
            keys.append((int(rec["HARPNUM"]), frame["raw_T_REC_TAI"]))
        frames_by_target[sid] = keys
        target_meta[sid] = rec

print("Structurally complete temporal targets:", len(frames_by_target))
print("Missing/incomplete relative to role table:", len(selected - set(frames_by_target)))


## 4. Scan the cached SHARP source once for the exact requested keys

In [ ]:
requested = {}
for sid, keys in frames_by_target.items():
    rec = target_meta[sid]
    for key in keys:
        requested[key] = {
            "sid": sid,
            "HARPNUM": int(rec["HARPNUM"]),
            "NOAA_AR_clean": int(rec["NOAA_AR_clean"]),
        }

raw = {}
match_count = defaultdict(int)
quality = {}
noaa_conflict = {}

with sharp_path.open("r", encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)
    cols = reader.fieldnames or []
    missing_features = [c for c in FEATURES + ["T_REC","HARPNUM"] if c not in cols]
    if missing_features:
        raise RuntimeError(f"Missing SHARP columns: {missing_features}")

    for row in reader:
        try:
            harp = int(float(row["HARPNUM"]))
        except Exception:
            continue
        key = (harp, row["T_REC"])
        if key not in requested:
            continue

        match_count[key] += 1

        vals = []
        for feat in FEATURES:
            try:
                v = float(row[feat])
            except Exception:
                v = np.nan
            vals.append(v if math.isfinite(v) else np.nan)
        raw[key] = np.asarray(vals, dtype=np.float64)

        q = row.get("QUALITY", "")
        try:
            q0 = int(float(q)) == 0
        except Exception:
            q0 = False
        quality[key] = q0

        explicit = str(row.get("NOAA_AR_clean", "")).strip()
        conflict = False
        if explicit and explicit.lower() not in {"nan","none","null"}:
            try:
                conflict = int(float(explicit)) != requested[key]["NOAA_AR_clean"]
            except Exception:
                conflict = True
        noaa_conflict[key] = conflict

print("Requested unique SHARP keys:", len(requested))
print("Matched unique SHARP keys:", len(raw))
print("Duplicate-match keys:", sum(v != 1 for v in match_count.values()))


## 5. Build the independent-test tensor under the conservative quality rule

In [ ]:
test_lookup = test.set_index(sid_col)
X_list, y_list, meta_rows = [], [], []

for sid in sorted(frames_by_target):
    keys = frames_by_target[sid]
    if any(match_count.get(k, 0) != 1 for k in keys):
        continue
    if any(k not in raw for k in keys):
        continue
    if any(not quality.get(k, False) for k in keys):
        continue
    if any(noaa_conflict.get(k, True) for k in keys):
        continue

    arr = np.stack([raw[k] for k in keys], axis=0)
    if arr.shape != (3, 15) or not np.isfinite(arr).all():
        continue

    r = test_lookup.loc[sid]
    X_list.append(arr)
    y_list.append(int(r[label_col]))
    meta_rows.append({
        "target_sample_id": sid,
        "HARPNUM": int(float(r["HARPNUM"])),
        "NOAA_AR_clean": int(float(r["NOAA_AR_clean"])),
        "region_component_id": r["region_component_id"],
        "stored_year": int(r[year_col]) if year_col else int(sid[:4]),
    })

X = np.asarray(X_list, dtype=np.float64)
y = np.asarray(y_list, dtype=np.int64)
meta = pd.DataFrame(meta_rows)

print("Final Cycle-25 test tensor:", X.shape)
print("Final labels:", y.shape)
print("Final positives:", int(y.sum()))
print("Final regions:", meta["region_component_id"].nunique())
print(meta["stored_year"].value_counts().sort_index().to_string())

if X.ndim != 3 or X.shape[1:] != (3,15):
    raise RuntimeError("Unexpected test tensor shape.")
if not np.isfinite(X).all():
    raise RuntimeError("Nonfinite values remain in final test tensor.")


## 6. Save arrays and immutable metadata for the frozen-model evaluation

In [ ]:
np.save(OUT / "sharp_cycle25_test_X.npy", X)
np.save(OUT / "sharp_cycle25_test_y.npy", y)
meta.to_csv(OUT / "sharp_cycle25_test_rows.csv.gz", index=False, compression="gzip")

protocol = {
    "status": "CYCLE25_SHARP_TEST_ARRAYS_BUILT_NO_MODEL_EVALUATION",
    "years": [2021,2022,2023,2024,2025],
    "history_minutes": [-288,-192,-96],
    "feature_order": FEATURES,
    "shape": list(X.shape),
    "positives": int(y.sum()),
    "regions": int(meta["region_component_id"].nunique()),
    "quality_rule": "QUALITY=0 at all three records, 15/15 finite features, one exact source match, no explicit NOAA conflict",
    "model_fitted": False,
    "calibration_fitted": False,
    "threshold_selected": False,
    "cycle25_used_for_tuning": False,
}
(OUT / "protocol_record.json").write_text(json.dumps(protocol, indent=2) + "\n")

print("OUTPUT:", OUT)
print("STATUS: CYCLE25_SHARP_TEST_ARRAYS_BUILT_NO_MODEL_EVALUATION")
